In [19]:
import pandas as pd
import numpy as np

# --- Загрузка данных ---
# Файл с предсказаниями: колонки без названий, поэтому задаём имена сами
df_pred = pd.read_csv("x_test_pred.csv", names=["id", "pred"], header=0)  # header=0 если есть заголовок? Уточним
# Если в файле нет заголовка, используйте header=None
# df_pred = pd.read_csv("x_test_pred.csv", names=["id", "pred"], header=None)

# Файл Б с реальными высотами
df_true = pd.read_csv("cup_it_example_src_B_cleaned.csv", usecols=["height_numeric"])
df_true = df_true.reset_index().rename({"index": "id"}, axis=1)

# --- Объединение по id (только те, для которых есть предсказания) ---
merged = pd.merge(df_true, df_pred, on="id", how="inner")

# Убедимся, что высоты числовые
merged["height_numeric"] = pd.to_numeric(merged["height_numeric"], errors="coerce")
merged["pred"] = pd.to_numeric(merged["pred"], errors="coerce")

# Удаляем строки с пропусками, если они есть
merged = merged.dropna(subset=["height_numeric", "pred"])

# Вычисляем разницу (предсказание - реальность)
merged["diff"] = merged["pred"] - merged["height_numeric"]

# --- Создание интервалов по реальной высоте ---
# Задаём границы интервалов. Например, от 0 до 100 с шагом 10.
# При необходимости измените bins под ваши данные (можно вычислить автоматически)
bins = list(range(0, 101, 10))  # [0, 10, 20, ..., 100]
labels = [f"{bins[i]}-{bins[i+1]}" for i in range(len(bins)-1)]

# Добавляем колонку с интервалом
merged["height_interval"] = pd.cut(merged["height_numeric"], bins=bins, labels=labels, right=False)

# --- Группировка по интервалам и расчёт метрик ---
def rmse(x):
    return np.sqrt(np.mean(x ** 2))

metrics = merged.groupby("height_interval").agg(
    count=("id", "count"),
    MAE=("diff", lambda x: np.abs(x).mean()),
    RMSE=("diff", rmse),
    mean_diff=("diff", "mean")  # средняя разница (смещение)
).reset_index()

# Выводим результат 
print(metrics)

# Сохраняем в CSV для дальнейшего анализа
metrics.to_csv("оценка_по_интервалам.csv", index=False)

# Дополнительно: можно вывести общую статистику для всех зданий
#print("\nОбщая статистика для всей выборки:")
#print(f"Количество зданий: {len(merged)}")
#print(f"Общая MAE: {np.abs(merged['diff']).mean():.2f}")
#print(f"Общая RMSE: {np.sqrt((merged['diff']**2).mean()):.2f}")`

  height_interval  count        MAE       RMSE  mean_diff
0            0-10   5226   2.775180   4.487511   2.323128
1           10-20   5129   2.180857   3.380644  -0.213459
2           20-30   1671   4.286340   6.202711  -1.681833
3           30-40   2495   2.781222   5.262862  -0.392311
4           40-50    770   4.096339   7.359651  -1.214681
5           50-60    698   4.967930   8.751894  -3.187855
6           60-70    154  10.042367  17.719033  -6.751132
7           70-80    139  10.774385  17.486563  -8.473138
8           80-90    180   9.865033  16.777094  -8.687193
9          90-100     64   3.744799   6.770624  -2.943350


In [14]:
df_true.reset_index().rename({"index": "id"}, axis=1)

,id,height_numeric
0,0,28.0
1,1,23.0
2,2,49.0
3,3,15.0
4,4,6.6
...,...,...
82621,82621,6.6
82622,82622,6.6
82623,82623,6.6
82624,82624,6.6


In [4]:
import pandas as pd
import numpy as np

# --- Загрузка данных ---
# Файл с предсказаниями: колонки без названий, поэтому задаём имена сами
df_pred = pd.read_csv("x_test_pred.csv", names=["id", "pred"], header=0)  # header=0 если есть заголовок? Уточним
# Если в файле нет заголовка, используйте header=None
# df_pred = pd.read_csv("x_test_pred.csv", names=["id", "pred"], header=None)

# Файл Б с реальными высотами
df_true = pd.read_csv("cup_it_example_src_B_cleaned.csv", usecols=["id", "height_numeric"])

In [5]:
df_true

,id,height_numeric
0,3,28.0
1,4,23.0
2,5,49.0
3,7,15.0
4,25,6.6
...,...,...
82621,161060,6.6
82622,161067,6.6
82623,161070,6.6
82624,161073,6.6
